In [1]:
import numpy as np
import pandas as pd

here also we should do text preprocessing part, because predict from user's input

In [2]:
import re
import string

In [3]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [4]:
#open the downloaded english stop words file
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines() #take line by line as a list

In [5]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [6]:
#create a function includeing all steps in preprocessing
def preprocessing(text):
    data = pd.DataFrame([text], columns=["tweet"])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].str.replace(r'\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    #data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

get the downloaded vocabulary

In [7]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [8]:
tokens

['fingerprint',
 'pregnanc',
 'test',
 'android',
 'app',
 'beauti',
 'cute',
 'health',
 'iger',
 'iphoneon',
 'iphonesia',
 'iphon',
 'final',
 'transpar',
 'silicon',
 'case',
 'thank',
 'uncl',
 'yay',
 'soni',
 'xperia',
 'sonyexperias…',
 'love',
 'would',
 'go',
 'talk',
 'makememori',
 'unplug',
 'relax',
 'smartphon',
 'wifi',
 'connect',
 'im',
 'wire',
 'know',
 'georg',
 'made',
 'way',
 'daventri',
 'home',
 'amaz',
 'servic',
 'appl',
 'wont',
 'even',
 'question',
 'unless',
 'pay',
 'stupid',
 'support',
 'softwar',
 'updat',
 'fuck',
 'phone',
 'big',
 'time',
 'happi',
 'us',
 'instap',
 'instadaili',
 'xperiaz',
 'new',
 'type',
 'c',
 'charger',
 'cabl',
 'uk',
 '…',
 'bay',
 'amazon',
 'etsi',
 'year',
 'rob',
 'cross',
 'tobi',
 'young',
 'evemun',
 'mcmafia',
 'taylor',
 'spectr',
 'newyear',
 'start',
 'recip',
 'technolog',
 'samsunggalaxi',
 'iphonex',
 'pictwittercompjiwqwtc',
 'bout',
 'shop',
 'listen',
 'music',
 'justm',
 'likeforlik',
 'followforfollow…'

vectorization

In [9]:
def vectorizer(ds, vocabulary):
    # Create an empty list to store the vectorized sentences
    vectorized_list = []

    # Loop through each sentence in the dataset
    for sentence in ds:
        # Create a vector of zeros with the same length as the vocabulary
        # Each position represents a word in the vocabulary
        sentence_list = np.zeros(len(vocabulary))

        # Loop through each word in the vocabulary
        for i in range(len(vocabulary)):

            # Check if the vocabulary word exists in the sentence
            # sentence.split() converts the sentence into a list of words
            if vocabulary[i] in sentence.split():

                # If the word is present, set the corresponding position to 1
                # This represents the presence of that word in the sentence
                sentence_list[i] = 1

        # After checking all vocabulary words, add the vector to the list
        vectorized_list.append(sentence_list)

    # Convert the list of vectors into a NumPy array with float32 datatype
    vectorized_list_new = np.asarray(vectorized_list, dtype=np.float32)

    # Return the vectorized dataset
    return vectorized_list_new

import model for predictions

In [10]:
import pickle

In [11]:
#open the model
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [12]:
with open('../static/model/tfidf_vectorizer.pickle', 'rb') as f:
    tfidf = pickle.load(f)

In [38]:
#take as positive or negative
def get_prediction(text):
    processed_series = preprocessing(text)
    processed_str = processed_series.iloc[0]
    
    vector = tfidf.transform([processed_str])
    
    proba_negative = model.predict_proba(vector)[0][1]
    threshold = 0.68          # Change to 0.60 or 0.68 if needed
    
    return "negative" if proba_negative >= threshold else "positive"

In [39]:
def debug_vectorization(text, vocab_list):
    # Get the processed result (it's a Series)
    processed_series = preprocessing(text)
    
    # Extract the actual string
    processed_str = processed_series.iloc[0]          # or processed_series[0]
    
    words = processed_str.split()
    vocab_set = set(vocab_list)                       # convert to set for fast lookup
    
    known = [w for w in words if w in vocab_set]
    unknown = [w for w in words if w not in vocab_set]
    
    print("Original text     :", text)
    print("After preprocessing:", processed_str)
    print("Words in sentence :", words)
    print("Known in vocab    :", known)
    print("Unknown / lost    :", unknown)
    print(f"Vocabulary coverage: {len(known)/max(1, len(words)):.1%}")
    print("-" * 60)

create a text for test

In [40]:
txt = 'not bad'

In [41]:
debug_vectorization(txt, tokens)

Original text     : not bad
After preprocessing: bad
Words in sentence : ['bad']
Known in vocab    : ['bad']
Unknown / lost    : []
Vocabulary coverage: 100.0%
------------------------------------------------------------


In [42]:
#preprocess the txt
preprocessed_txt = preprocessing(txt)

In [43]:
preprocessed_txt

0    bad
Name: tweet, dtype: str

In [44]:
vectorized_txt = vectorizer(preprocessed_txt, tokens)

In [45]:
vectorized_txt

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 15925), dtype=float32)

In [46]:
print("Number of words that became 1 in vector:", vectorized_txt.sum())
print("Total features in vocab:", vectorized_txt.shape[1])

Number of words that became 1 in vector: 1.0
Total features in vocab: 15925


In [53]:
prediction = get_prediction(txt)
print(prediction)

positive


another example

In [52]:
test_sentences = [
    "can be used for many works",
    "great product i like it",
    "this is okay for daily use",
    "worst phone ever",
    "eww too bad"
]

for sentence in test_sentences:
    print(f"Text: {sentence}")
    print(f"Prediction: {get_prediction(sentence)}")
    print("-" * 50)

Text: can be used for many works
Prediction: positive
--------------------------------------------------
Text: great product i like it
Prediction: positive
--------------------------------------------------
Text: this is okay for daily use
Prediction: negative
--------------------------------------------------
Text: worst phone ever
Prediction: negative
--------------------------------------------------
Text: eww too bad
Prediction: positive
--------------------------------------------------


In [35]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(classification_report(y_test, y_test_pred, target_names=["positive (0)", "negative (1)"]))

ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred,
                                        display_labels=["positive","negative"],
                                        cmap="Blues")
plt.show()

NameError: name 'y_test' is not defined

In [54]:
from transformers import pipeline

pipe = pipeline("text-classification", model="../static/model/distilbert-sentiment-final")

texts = [
    "not bad product",
    "this is okay for daily use",
    "battery dies in 2 hours smh",
    "great product i like it",
    "worst phone ever",
    "no problem at all",
    "disappointed",
    "trash",
    "surprisingly not bad"
]

for t in texts:
    res = pipe(t)[0]
    label = "positive" if res['label'] == 'LABEL_0' else "negative"
    print(f"{t:35} → {label} ({res['score']:.3f})")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '../static/model/distilbert-sentiment-final'. Use `repo_type` argument if needed.